# Exploration

Ce notebook présente la structure de base de l'entraînement du modèle.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from training.data import load_data

In [2]:
# On utilise la fonction load_data qui standardise le chargement des données pour éviter de répéter le code dans chaque notebook
df = load_data('../data/employees.csv')

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import f1_score, roc_auc_score

target = "LeaveOrNot"
X = df.drop(columns=[target])
y = df[target]

categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), categorical_features),
    ],
    remainder="passthrough",
)
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42)),
])

param_grid = {
    "classifier__n_estimators": [50, 100, 200],
    "classifier__max_depth": [5, 10],
}

grid_search = GridSearchCV(
    pipeline, param_grid, cv=5, scoring="accuracy", # n_jobs=-1
)
grid_search.fit(X_train, y_train)

test_accuracy = grid_search.score(X_test, y_test)
print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")
print(f"Test accuracy:    {test_accuracy:.4f}")

Best parameters: {'classifier__max_depth': 10, 'classifier__n_estimators': 200}
Best CV accuracy: 0.8436
Test accuracy:    0.8700


In [7]:
import mlflow, joblib

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("employee_attrition_notebook")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1782287062586, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1782287062586, lifecycle_stage='active', name='employee_attrition_notebook', tags={}, trace_location=None, workspace='default'>

In [8]:
with mlflow.start_run():
    grid_search.fit(X_train, y_train)
    mlflow.log_metric("test_f1", f1_score(y_test, grid_search.predict(X_test)))
    mlflow.log_metric("test_auc", roc_auc_score(y_test, grid_search.predict_proba(X_test)[:, 1]))
    joblib.dump(grid_search, "../models/employee_attrition_model.joblib")

🏃 View run clumsy-hog-284 at: http://127.0.0.1:5000/#/experiments/1/runs/7bd1a55816dc4f92b426463904d10de4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [9]:
import mlflow.sklearn

mlflow.sklearn.autolog()
with mlflow.start_run():
    grid_search.fit(X_train, y_train)
    mlflow.log_metric("test_f1", f1_score(y_test, grid_search.predict(X_test)))
    mlflow.log_metric("test_auc", roc_auc_score(y_test, grid_search.predict_proba(X_test)[:, 1]))
    joblib.dump(grid_search, "../models/employee_attrition_model.joblib")

2026/06/24 09:54:29 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\cepe-s4-form\Desktop\mlops-training\mlops-training\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/24 09:54:34 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\cepe-

🏃 View run treasured-hawk-950 at: http://127.0.0.1:5000/#/experiments/1/runs/58ca54b1ceea40aaadf39622846c359b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run powerful-stag-408 at: http://127.0.0.1:5000/#/experiments/1/runs/cfa5edf7fb7540ed8d37febd8ffd9145
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run bouncy-colt-826 at: http://127.0.0.1:5000/#/experiments/1/runs/09e653e06d6e4e15ae696004b8671952
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


2026/06/24 09:54:44 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\cepe-s4-form\Desktop\mlops-training\mlops-training\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/06/24 09:54:44 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\cepe-

🏃 View run bustling-goat-827 at: http://127.0.0.1:5000/#/experiments/1/runs/f83fb62ed3674110b27f8336c2becbae
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run gaudy-pig-884 at: http://127.0.0.1:5000/#/experiments/1/runs/004a0553cbe74efdbff34da9b58c1289
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run nervous-ram-0 at: http://127.0.0.1:5000/#/experiments/1/runs/a0d9c83666f3450fb48f5c500b4ebd57
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [11]:
mlflow.search_registered_models()

[<RegisteredModel: aliases={}, creation_timestamp=1782288508172, deployment_job_id='', deployment_job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', description=('Modèle qui détermine si un collaborateur a un risque de démissionner dans '
  "l'année."), last_updated_timestamp=1782288910156, latest_versions=[<ModelVersion: aliases=[], creation_timestamp=1782288910156, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1782288910156, metrics=None, model_id=None, name='employee_attrition', params=None, run_id='a0d9c83666f3450fb48f5c500b4ebd57', run_link='', source='models:/m-d21a2f726ea64450919406302727b5f5', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>], name='employee_attrition', tags={'domain': 'HR'}, workspace='default'>

## Exemple de connexion à un serveur MLFlow

In [12]:
from mlflow import MlflowClient

client = MlflowClient('http://127.0.0.1:5000')

In [13]:
experiment = client.get_experiment_by_name('employee_attrition_notebook')

In [30]:
last_run = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=['metrics.best_cv_score DESC'],
    max_results=1
)[0]

last_run.info.run_name

'nervous-ram-0'

In [31]:
model_name = "employee_attrition"
model_uri = f'runs:/{last_run.info.run_id}/best_estimator'

model_version = mlflow.register_model(model_uri, model_name)


Registered model 'employee_attrition' already exists. Creating a new version of this model...
2026/06/24 10:54:10 WARNING mlflow.tracking._model_registry.fluent: Run with id a0d9c83666f3450fb48f5c500b4ebd57 has no artifacts at artifact path 'best_estimator', registering model based on models:/m-d21a2f726ea64450919406302727b5f5 instead
2026/06/24 10:54:10 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: employee_attrition, version 4
Created version '4' of model 'employee_attrition'.


In [ ]:
import json

sample = X_test.head(1)
payload = json.dumps({"dataframe_split": sample.to_dict(orient="split")})

payload = json.dumps({"dataframe_split": sample.to_dict(orient="split")}).replace('"', '\\"')

print(f"curl.exe -X POST http://127.0.0.1:5001/invocations -H 'Content-Type: application/json' -d '{payload}'")

curl.exe -X POST http://127.0.0.1:5001/invocations -H 'Content-Type: application/json' -d '{\"dataframe_split\": {\"index\": [297], \"columns\": [\"Education\", \"JoiningYear\", \"City\", \"PaymentTier\", \"Age\", \"Gender\", \"EverBenched\", \"ExperienceInCurrentDomain\"], \"data\": [[\"Bachelors\", 2016, \"Bangalore\", 3, 24, \"Female\", \"No\", 2]]}}'


In [ ]:

print(f"curl.exe -X POST http://127.0.0.1:5001/invocations -H 'Content-Type: application/json' -d '{json.dumps(payload)}.replace(\'"\', '"')